In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import time
import random
from collections import defaultdict, Counter

def load_graph(path):
    if path.endswith('.csv'):
        try:
            df = pd.read_csv(path, sep=None, engine='python')
            if len(df.columns) < 2:
                raise ValueError("Dataset does not have at least 2 columns.")
            G = nx.from_pandas_edgelist(df, source=df.columns[0], target=df.columns[1])
        except Exception:
            # Fallback for heavily malformed CSVs
            G = nx.read_edgelist(path, delimiter=',', comments='#')
    else:
        G = nx.read_edgelist(path, comments='#')

    G.remove_edges_from(nx.selfloop_edges(G))
    G = nx.convert_node_labels_to_integers(G)
    return G

def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def generate_continuous_encoding(G):
    x_dict = {}
    for node in G.nodes():
        deg = len(list(G.neighbors(node)))
        # Vector bounds [-10, 10] per paper's equation 14
        x_dict[node] = np.random.uniform(-10, 10, deg + 1)
    return x_dict

def ANCE(G, x_dict):
    """Algorithm 1: Attribute Network Continuous Encoding Method"""
    L1_edges = []

    for node in G.nodes():
        neighbors = list(G.neighbors(node))
        d_i = len(neighbors)
        if d_i == 0: continue

        x_i = x_dict[node]
        h_i = sigmoid(x_i)
        p_i = softmax(h_i)

        p_i_edges = p_i[:-1]
        s1_idx = np.argmax(p_i_edges)
        L1_edges.append((node, neighbors[s1_idx]))

    G1 = nx.Graph()
    G1.add_nodes_from(G.nodes())
    G1.add_edges_from(L1_edges)
    label1 = {node: min(comp) for comp in nx.connected_components(G1) for node in comp}

    return label1

def compute_objectives(G, label_dict, Atr):
    """Evaluates the two objectives: f1 (Negative Modularity) and f2 (Attribute Penalty)"""
    comm_dict = defaultdict(list)
    for node, c_id in label_dict.items():
        comm_dict[c_id].append(node)
    partition = list(comm_dict.values())

    # f1: Network Structure (Negative Modularity)
    if len(partition) <= 1:
        f1 = 1.0
    else:
        try:
            f1 = -nx.algorithms.community.quality.modularity(G, partition)
        except ZeroDivisionError:
            f1 = 1.0

    # f2: Node Attribute Similarity (Penalty for mismatched attributes)
    f2_penalty = 0.0
    count = 0
    for comp in partition:
        if len(comp) > 1:
            attrs = [Atr[n] for n in comp]
            most_common_count = Counter(attrs).most_common(1)[0][1]
            f2_penalty += (len(comp) - most_common_count)
            count += len(comp)

    f2 = f2_penalty / (count + 1e-6)

    return np.array([f1, f2])

def DE_crossover_mutation(x1, x2, x3, F_weight=0.7, CR=0.4):
    """Differential Evolution and Polynomial Mutation operators"""
    y = {}
    for node in x1.keys():
        mutant = x1[node] + F_weight * (x2[node] - x3[node])

        # Binomial Crossover
        mask = np.random.rand(len(mutant)) < CR
        if not np.any(mask):
            mask[np.random.randint(len(mask))] = True
        offspring = np.where(mask, mutant, x1[node])

        # Polynomial Mutation Approximation (Gaussian noise)
        pm_mask = np.random.rand(len(offspring)) < 0.01
        noise = np.random.normal(0, 1.0, len(offspring))
        offspring = np.where(pm_mask, offspring + noise, offspring)

        # Bounding to [-10, 10]
        y[node] = np.clip(offspring, -10.0, 10.0)
    return y

def CEMOV(G, Atr, pop_size=10, generations=5):
    """
    Algorithm 2: CEMOV (Continuous Encoding MOEA/D)
    """
    # 1. Initialize Population
    population = [generate_continuous_encoding(G) for _ in range(pop_size)]
    weights = np.array([[i/(pop_size-1), 1.0 - i/(pop_size-1)] for i in range(pop_size)])

    # Neighborhood definition (T-size)
    t_size = max(2, pop_size // 3)
    B = [[(i+j)%pop_size for j in range(-t_size//2, t_size//2 + 1)] for i in range(pop_size)]

    # 2. Evaluate Initial Population & Reference Point
    obj_values = np.zeros((pop_size, 2))
    for i in range(pop_size):
        label1 = ANCE(G, population[i])
        obj_values[i] = compute_objectives(G, label1, Atr)

    z_star = np.min(obj_values, axis=0)

    # 3. Evolution Loop
    for gen in range(generations):
        for i in range(pop_size):
            # Mating Selection
            pool = B[i] if np.random.rand() < 0.9 else list(range(pop_size))

            # Ensure we have at least 3 unique indices for DE
            if len(pool) >= 3:
                idx1, idx2, idx3 = np.random.choice(pool, 3, replace=False)
            else:
                idx1, idx2, idx3 = np.random.choice(list(range(pop_size)), 3, replace=False)

            # Reproduction (DE + PM)
            y = DE_crossover_mutation(population[idx1], population[idx2], population[idx3])

            # Evaluation
            label1_y = ANCE(G, y)
            obj_y = compute_objectives(G, label1_y, Atr)

            # Update Reference Point
            z_star = np.minimum(z_star, obj_y)

            # Tchebycheff Neighborhood Update
            c_r = 0
            n_r = 2 # Max replacement limit

            for j in pool:
                g_old = np.max(weights[j] * np.abs(obj_values[j] - z_star))
                g_new = np.max(weights[j] * np.abs(obj_y - z_star))

                if g_new < g_old:
                    population[j] = y
                    obj_values[j] = obj_y
                    c_r += 1
                if c_r >= n_r:
                    break

    # Find the best individual based on modularity (-f1)
    best_idx = np.argmin(obj_values[:, 0])
    best_modularity = -obj_values[best_idx, 0]

    # Calculate the number of communities for the best individual
    best_individual = population[best_idx]
    best_label = ANCE(G, best_individual)
    num_communities = len(set(best_label.values()))

    return best_modularity, num_communities

# --- Execution Block ---
datasets = [
    "/content/Dataset_CyberCrime_Sean.csv",
    "/content/london_crime_by_lsoa.csv",
    "/content/twitter_combined.txt.gz",
    "/content/Meetings.csv",
    "/content/Phone_Calls.csv",
    "/content/Roles.csv",
    "/content/Sicilian Mafia.csv",
    "/content/facebook_combined.txt.gz"
]

print(f"{'Dataset':<32} | {'Nodes':<6} | {'Edges':<7} | {'Comms':<6} | {'Modularity':<10} | {'Runtime (s)':<12}")
print("-" * 88)

for ds in datasets:
    try:
        start_time = time.time()

        G = load_graph(ds)
        nodes_count = G.number_of_nodes()
        edges_count = G.number_of_edges()

        # Generate random discrete attributes for testing
        Atr = {node: random.randint(0, 2) for node in G.nodes()}

        # Run Algorithm 2 (CEMOV)
        best_modularity, num_communities = CEMOV(G, Atr, pop_size=10, generations=5)

        runtime = time.time() - start_time
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {nodes_count:<6} | {edges_count:<7} | {num_communities:<6} | {best_modularity:<10.4f} | {runtime:<12.4f}")

    except FileNotFoundError:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'N/A':<10} | {'N/A':<12}")
    except Exception as e:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {str(e)[:10]:<10} | {'N/A':<12}")

Dataset                          | Nodes  | Edges   | Comms  | Modularity | Runtime (s) 
----------------------------------------------------------------------------------------
Dataset_CyberCrime_Sean.csv      | 136    | 161     | 20     | 0.6306     | 0.9251      
london_crime_by_lsoa.csv         | 4868   | 4835    | 33     | 0.9676     | 149.2188    
twitter_combined.txt.gz          | 81306  | 1342296 | 1494   | 0.3115     | 2056.0246   
Meetings.csv                     | 95     | 247     | 11     | 0.5141     | 0.3965      
Phone_Calls.csv                  | 92     | 119     | 10     | 0.5989     | 0.3626      
Roles.csv                        | 161    | 134     | 27     | 0.7742     | 0.6420      
Sicilian Mafia.csv               | 143    | 325     | 15     | 0.5276     | 0.5743      
facebook_combined.txt.gz         | 4039   | 88234   | 108    | 0.4755     | 24.8508     
